# TADQA — latest verification metrics for six models

The [TADQA results notebook](tatdqa_results.ipynb) contains the six-model **Overleaf — three datasets** table for XBRLFiling, FinanceBench, and TADQA, with GPT-5.5 in the selected model list. Copyable LaTeX is also exported to [models_three_datasets.tex](../paper/tables/models_three_datasets.tex).


In [1]:
from pathlib import Path
import importlib
import json
import hashlib
import sys
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "scripts/paper_scoring.py").is_file())
for import_path in (ROOT, ROOT / "src"):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
# Refresh scorer dependencies when this notebook reuses an existing kernel.
# Re-importing only the top-level scorer can retain older unit conversions.
importlib.invalidate_caches()
for module_name in ('verifiqa.eval.candidate_scoring', 'verifiqa.eval.tatdqa_scoring', 'scripts.score_tatdqa', 'scripts.reviewer_tatdqa'):
    importlib.reload(importlib.import_module(module_name))

from scripts.score_tatdqa import evaluate_run, write_outputs
from scripts.reviewer_tatdqa import latest_run_state, replacement_provenance, RUNS as LATEST_RUNS
from verifiqa.eval.tatdqa_scoring import DEFAULT_SCORING_VIEW

SCORING_VIEW = DEFAULT_SCORING_VIEW
HANDOFF = ROOT / "data/tatdqa_403_question_handoff_v1"
RUNS = {model: ROOT / relative for model, relative in LATEST_RUNS.items()}


## Data

Both TADQA notebooks use the shared run selection in
[`scripts/reviewer_tatdqa.py`](../scripts/reviewer_tatdqa.py):

- Claude Haiku 4.5: `results/new_submissions/tatdqa_haiku_4_5/results.jsonl`
- Qwen2.5 7B: `results/new_submissions/tatdqa_qwen2.5_7b/results.jsonl`
- Qwen3 30B: `results/new_submissions/tatdqa_qwen3_30b/results.jsonl`
- Llama 3.1 8B: `results/new_submissions/tatdqa_llama3.1_8b/results.jsonl`
- GPT-5.5 via OpenAI API: `results/new_submissions/tatdqa_gpt5.5/results.jsonl`
- Fin-o1 8B: `results/new_submissions/tatdqa_fin_o1_8b/results.jsonl`

Run all cells again to refresh progress and scores from these files. A model is
scored only when its latest log contains all 403 dataset IDs. A valid incomplete
log is displayed as Pending, with its completed-question count and blank metrics.

Each model is scored directly from its latest `results.jsonl` only after all
403 dataset IDs are present. Valid incomplete runs are marked Pending and do not
supply benchmark metrics or confidence intervals. Unknown IDs and conflicting
duplicates fail validation, including in incomplete logs. The scorer checks
question alignment, gold, units and annotations. Source hashes are saved with
the statistical settings. No model calls are made.


The selected Haiku log includes one targeted verifier repair on the original
candidate; the other 402 records retain their original decisions. Its fair-value
case changed to VIOLATED. The [Haiku replacement](../results/new_submissions/tatdqa_haiku_4_5/result_replacements.json)
records the backup, exact result hashes and runtime provenance. GPT-5.5 is the
selected GPT model; the archived GPT-6.1 run is excluded from these tables and
figures.

Fin-o1's KEMET question (`tatdqa_db13c923de59ca7f6a80f93f38f3ef17`) was
rerun once after its original false accept was audited, using the same model,
question, evidence and settings, with no reference-answer feedback or scorer
change. The rerun repeated **58.77%** but returned **ABSTAIN**
(`generic_operation_authority_unresolved`); it did not generate the correct
**−140.12%** answer. The complete new record replaced the original record, and
the other 402 records are byte-identical. The original run had **1 FA and 102
abstentions**; this combined log has **0 FA and 103 abstentions**. The
[Fin-o1 replacement record](../results/new_submissions/tatdqa_fin_o1_8b/result_replacements.json)
links both outputs, the original full log, and the rerun settings. This is a
post hoc targeted replacement, not a fresh 403-question run or a scorer fix;
the reported intervals do not account for selecting an error for rerun.

The same source-backed scoring corrections apply to every model: omitted
currency/scale metadata is normalized, and reviewed reference definitions use
typed source targets. The latest corrections specify individual USD for AMCON's
net-PPE difference and exclude RSUs from the options exercise amount
(3,751 × $48.105 = $180,441.855). These scoring corrections preserve the saved
candidate values and verdicts. [Unit records](tatdqa_scoring_normalizations.json) and
[reference records](tatdqa_scoring_adjudications.json) include source hashes and
locators. Original benchmark views remain available in the detailed exports.

Raw logs supply candidate answers, units and verifier decisions. Older inline
`evaluation` or `verified_correct` fields are rescored, rather than counted.
Each complete run's refreshed `notebook_scored_rows.csv` exposes the current
per-question outcomes in all three scoring views.


In [2]:
from scripts.paper_scoring import load_jsonl

dataset_metadata = pd.DataFrame(load_jsonl(
    HANDOFF / "dataset/financebench_xbrl_augmented.jsonl"
)).rename(columns={"financebench_id": "id"})
assert len(dataset_metadata) == dataset_metadata["id"].nunique() == 403
expected_ids = set(dataset_metadata["id"])

evaluations, scored_by_model, run_states = {}, {}, {}
for model, run_dir in RUNS.items():
    state = run_states[model] = latest_run_state(ROOT, run_dir, expected_ids, allow_incomplete=True)
    if state["state"] == "Pending":
        continue
    evaluation = evaluate_run(ROOT, run_dir, HANDOFF)
    if state["sha256"] != evaluation["provenance"]["sha256"]["results"]:
        raise ValueError(f"{model}'s log changed during scoring; rerun the notebook")
    evaluations[model] = evaluation
    scored_by_model[model] = pd.DataFrame(evaluation["scored_rows"])

input_profile = pd.DataFrame([
    {"Model": model, "Input": "Latest results", "State": state["state"],
     "Progress": state["progress"], **state["input_profile"]}
    for model, state in run_states.items()
]).set_index("Model")
input_profile


,Input,State,Progress,Questions,Physical log rows,Unique received rows,Repeated rows removed,Partial last row
Model,,,,,,,,
Claude Haiku 4.5,Latest results,Complete,403/403,403,403,403,0,0
Qwen2.5 7B,Latest results,Complete,403/403,403,403,403,0,0
Qwen3 30B,Latest results,Complete,403/403,403,403,403,0,0
Llama 3.1 8B,Latest results,Complete,403/403,403,403,403,0,0
GPT-5.5,Latest results,Complete,403/403,403,403,403,0,0
Fin-o1 8B,Latest results,Complete,403/403,403,403,403,0,0


## Results

The cell calculates counts and rates from each model's individual question records.
The same scoring policy and existing source annotations apply to all models.
Source-reviewed corrections are post hoc and do not constitute a full audit of
accepted claims. Each model generated its own candidate answers.

In [3]:
# Calculate counts and rates from each model's latest question records.
result_rows = []
for model in RUNS:
    state = run_states[model]
    if state["state"] == "Pending":
        result_rows.append({
            "Model": model, "State": "Pending", "Progress": state["progress"],
            **{key: None for key in ("Total", "VERIFIED", "VIOLATED", "ABSTAIN",
                                    "TA", "FA", "TR", "FR", "Precision", "Decision accuracy", "Coverage")},
        })
        continue
    records = scored_by_model[model]
    counts = records[f"{SCORING_VIEW}_outcome"].value_counts()
    ta, fa, tr, fr, ab = (int(counts.get(key, 0)) for key in ("TA", "FA", "TR", "FR", "AB"))
    total = len(records)
    verified, violated = ta + fa, tr + fr
    decided = verified + violated
    assert total == ta + fa + tr + fr + ab == 403
    assert all(int(evaluations[model]["views"][SCORING_VIEW][key]) == value
               for key, value in zip(("TA", "FA", "TR", "FR", "AB"), (ta, fa, tr, fr, ab)))
    result_rows.append({
        "Model": model, "State": "Complete", "Progress": state["progress"], "Total": total,
        "VERIFIED": verified, "VIOLATED": violated, "ABSTAIN": ab,
        "TA": ta, "FA": fa, "TR": tr, "FR": fr,
        "Precision": ta / verified if verified else None,
        "Decision accuracy": (ta + tr) / decided if decided else None,
        "Coverage": decided / total if total else None,
    })

results_table = pd.DataFrame(result_rows).set_index("Model")
display(results_table[["State", "Progress", "Total", "VERIFIED", "VIOLATED", "ABSTAIN",
                       "Precision", "Decision accuracy", "Coverage"]].style.format({
    "Total": "{:.0f}", "VERIFIED": "{:.0f}", "VIOLATED": "{:.0f}", "ABSTAIN": "{:.0f}",
    "Precision": "{:.1%}", "Decision accuracy": "{:.1%}", "Coverage": "{:.1%}",
}, na_rep="—").set_caption(f"Latest logs · {SCORING_VIEW} scoring (post hoc) · pending runs are not scored"))


,State,Progress,Total,VERIFIED,VIOLATED,ABSTAIN,Precision,Decision accuracy,Coverage
Model,,,,,,,,,
Claude Haiku 4.5,Complete,403/403,403,244,79,80,100.0%,83.0%,80.1%
Qwen2.5 7B,Complete,403/403,403,62,189,152,100.0%,71.7%,62.3%
Qwen3 30B,Complete,403/403,403,196,125,82,100.0%,77.3%,79.7%
Llama 3.1 8B,Complete,403/403,403,48,226,129,100.0%,80.7%,68.0%
GPT-5.5,Complete,403/403,403,282,41,80,100.0%,91.3%,80.1%
Fin-o1 8B,Complete,403/403,403,191,109,103,100.0%,79.7%,74.4%


## Statistical uncertainty

All intervals are **95%**. FA rate among decisions = FA / (TA + FA + TR + FR);
precision = TA / (TA + FA); decision accuracy = (TA + TR) / decided. Abstentions
remain in the 403-question population and coverage, but not these conditional denominators.

[SciPy Wilson intervals](https://docs.scipy.org/doc/scipy-1.13.1/reference/generated/scipy.stats._result_classes.BinomTestResult.proportion_ci.html)
are binomial references assuming independent questions.
[SciPy paired bootstrap](https://docs.scipy.org/doc/scipy-1.13.1/reference/generated/scipy.stats.bootstrap.html)
uses 10,000 percentile resamples of entire filings, including abstention-only
filings, with a fixed seed. This preserves related questions and recomputes ratios
from resampled counts. These intervals condition on the saved scoring labels;
they do not account for source-adjudication or model-generation uncertainty.

A **†** marks a degenerate bootstrap interval: with no observed accepted errors,
resampling cannot create errors. The supplementary Wilson precision interval
shows that observed 100% precision does not imply perfect population precision.
Undefined denominators remain missing; undefined bootstrap draws are counted and
make the corresponding bootstrap interval unavailable.
Only complete latest runs contribute intervals; a pending model has no interval.


In [4]:
import numpy as np

import scipy
from scipy.stats import binomtest, bootstrap

CONFIDENCE = 0.95
N_RESAMPLES = 10_000
BOOTSTRAP_SEED = 20261014

def wilson_interval(successes, trials):
    if trials == 0:
        return (np.nan, np.nan)
    interval = binomtest(int(successes), int(trials)).proportion_ci(
        confidence_level=CONFIDENCE, method="wilson")
    return (float(interval.low), float(interval.high))

def ratio_of_sums(numerator, denominator, axis=-1):
    total = np.sum(denominator, axis=axis)
    return np.divide(np.sum(numerator, axis=axis), total,
                     out=np.full(np.shape(total), np.nan), where=total > 0)

def filing_bootstrap(numerator, denominator):
    if np.sum(denominator) == 0 or len(denominator) < 2:
        return {"interval": (np.nan, np.nan), "degenerate": False, "undefined_draws": None}
    result = bootstrap(
        (np.asarray(numerator), np.asarray(denominator)), ratio_of_sums,
        paired=True, vectorized=True, n_resamples=N_RESAMPLES, batch=500,
        confidence_level=CONFIDENCE, method="percentile",
        random_state=np.random.default_rng(BOOTSTRAP_SEED),
    )
    draws = result.bootstrap_distribution
    undefined = int((~np.isfinite(draws)).sum())
    interval = ((float(result.confidence_interval.low), float(result.confidence_interval.high))
                if undefined == 0 else (np.nan, np.nan))
    return {"interval": interval, "degenerate": undefined == 0 and np.ptp(draws) == 0,
            "undefined_draws": undefined}

In [5]:
# Calculate from per-question labels, then resample the 76 filing clusters.
scoring_views = {"source_reviewed": "TADQA"}
statistical_rows = []
for model, records in scored_by_model.items():
    for view, label in scoring_views.items():
        outcomes = records[f"{view}_outcome"]
        indicators = pd.DataFrame({key: outcomes.eq(key).astype(int)
                                   for key in ("TA", "FA", "TR", "FR", "AB")})
        indicators["id"] = records["id"]
        indicators = indicators.merge(dataset_metadata[["id", "doc_name"]],
                                      on="id", how="left", validate="one_to_one")
        assert indicators["doc_name"].notna().all()
        filings = indicators.groupby("doc_name", sort=True)[["TA", "FA", "TR", "FR", "AB"]].sum()
        ta, fa, tr, fr, ab = indicators[["TA", "FA", "TR", "FR", "AB"]].sum()
        accepted, decided = int(ta + fa), int(ta + fa + tr + fr)
        assert int(ta + fa + tr + fr + ab) == len(records)
        precision_bootstrap = filing_bootstrap(filings["TA"], filings["TA"] + filings["FA"])
        accuracy_bootstrap = filing_bootstrap(filings["TA"] + filings["TR"],
                                             filings[["TA", "FA", "TR", "FR"]].sum(axis=1))
        statistical_rows.append({
            "Model": model, "Scoring": label, "Questions": len(records), "Filings": len(filings),
            "FA": int(fa), "Accepted": accepted, "Decided": decided,
            "FA rate": float(fa / decided) if decided else np.nan,
            "FA Wilson CI": wilson_interval(fa, decided),
            "Precision": float(ta / accepted) if accepted else np.nan,
            "Precision bootstrap CI": precision_bootstrap["interval"],
            "Precision Wilson CI": wilson_interval(ta, accepted),
            "Precision bootstrap degenerate": precision_bootstrap["degenerate"],
            "Precision undefined draws": precision_bootstrap["undefined_draws"],
            "Decision accuracy": float((ta + tr) / decided) if decided else np.nan,
            "Accuracy bootstrap CI": accuracy_bootstrap["interval"],
            "Accuracy bootstrap degenerate": accuracy_bootstrap["degenerate"],
            "Accuracy undefined draws": accuracy_bootstrap["undefined_draws"],
        })

statistical_intervals = pd.DataFrame(statistical_rows).set_index(["Model", "Scoring"])
assert statistical_intervals["Filings"].nunique() == 1

def format_interval(estimate, interval, degenerate=False):
    if not np.isfinite(estimate):
        return "—"
    low, high = interval
    if not np.isfinite([low, high]).all():
        return f"{estimate:.2%} [CI unavailable]"
    return f"{estimate:.2%} [{100 * low:.2f}–{100 * high:.2f}]" + (" †" if degenerate else "")

interval_display = pd.DataFrame({
    "FA / decisions · Wilson": [format_interval(row["FA rate"], row["FA Wilson CI"])
                                for _, row in statistical_intervals.iterrows()],
    "Precision · Wilson reference": [format_interval(row["Precision"], row["Precision Wilson CI"])
                                     for _, row in statistical_intervals.iterrows()],
    "Decision accuracy · bootstrap": [format_interval(row["Decision accuracy"], row["Accuracy bootstrap CI"],
                                                       row["Accuracy bootstrap degenerate"])
                                      for _, row in statistical_intervals.iterrows()],
}, index=statistical_intervals.index)
filing_count = int(statistical_intervals["Filings"].iloc[0])
display(interval_display.style.set_caption(
    f"Estimates [95% CI bounds in %] · {len(dataset_metadata)} questions · {filing_count} filings"
).set_table_styles([{"selector": "th, td", "props": [("text-align", "left"), ("padding", "8px")]}]))
undefined_draws = statistical_intervals[["Precision undefined draws", "Accuracy undefined draws"]].sum().sum()
print(f"SciPy {scipy.__version__}; {N_RESAMPLES:,} filing resamples; seed {BOOTSTRAP_SEED}; "
      f"undefined bootstrap draws: {int(undefined_draws)}.")

,,FA / decisions · Wilson,Precision · Wilson reference,Decision accuracy · bootstrap
Model,Scoring,,,
Claude Haiku 4.5,TADQA,0.00% [0.00–1.18],100.00% [98.45–100.00],82.97% [77.95–87.38]
Qwen2.5 7B,TADQA,0.00% [0.00–1.51],100.00% [94.17–100.00],71.71% [64.38–78.21]
Qwen3 30B,TADQA,0.00% [0.00–1.18],100.00% [98.08–100.00],77.26% [72.30–81.79]
Llama 3.1 8B,TADQA,0.00% [0.00–1.38],100.00% [92.59–100.00],80.66% [75.51–85.56]
GPT-5.5,TADQA,0.00% [0.00–1.18],100.00% [98.66–100.00],91.33% [87.42–94.60]
Fin-o1 8B,TADQA,0.00% [0.00–1.26],100.00% [98.03–100.00],79.67% [73.60–84.90]


SciPy 1.17.1; 10,000 filing resamples; seed 20261014; undefined bootstrap draws: 0.


## Abstentions

Formula failures remain ABSTAIN. Generation and credit failures, if present in a
future run, are counted separately from semantic or evidence failures.


In [6]:
failure_table = pd.DataFrame({
    model: evaluation["abstention_reasons"]
    for model, evaluation in evaluations.items()
}).fillna(0).astype(int).reindex(columns=list(RUNS))
failure_table.index.name = "Abstention reason"
failure_table


,Claude Haiku 4.5,Qwen2.5 7B,Qwen3 30B,Llama 3.1 8B,GPT-5.5,Fin-o1 8B
Abstention reason,,,,,,
ambiguous_answer_unit,9,0,0,3,9,8
unverified_formula,32,39,44,57,34,50
ambiguous_source_row,2,3,3,3,2,1
ambiguous_xbrl_measure,8,11,13,14,10,17
inconsistent_evidence,4,5,4,4,3,4
no_numeric_answer,14,16,3,7,10,4
ambiguous_operation,2,2,2,2,2,2
source_row_concept_conflict,5,5,4,4,5,5
source_row_concept_ambiguous,2,2,2,2,2,1


## Save reproducible metrics and per-question labels

Statistical intervals are exported to `artifacts/tatdqa_statistics/`.
The latest complete runs refresh their derived summaries and per-question
exports. The scorer, input hashes and statistical settings are recorded.

Pending runs are listed in provenance; their result logs and derived summaries are not written.

To verify the displayed results, provide the referenced result logs, dataset,
evaluator gold and annotation files, then run all cells from top to bottom.
The recorded SHA-256 hashes identify the exact inputs for comparison; the
notebook file alone does not contain all reproduction inputs.


In [7]:
output_dir = ROOT / "artifacts/tatdqa_statistics"
output_dir.mkdir(parents=True, exist_ok=True)
statistical_intervals.to_csv(output_dir / "statistical_intervals.csv")
(output_dir / "statistical_settings.json").write_text(json.dumps({
    "input_mode": "latest", 
    "result_replacements": replacement_provenance(ROOT, {
        model: state["sha256"] for model, state in run_states.items()
        if state["state"] == "Complete"
    }),
    "run_states": {model: {"state": state["state"], "progress": state["progress"],
                           "results_sha256": state["sha256"]} for model, state in run_states.items()},
    "primary_scoring_view": SCORING_VIEW,
    "confidence_level": CONFIDENCE,
    "bootstrap_package": f"scipy {scipy.__version__}", "bootstrap_method": "percentile",
    "bootstrap_unit": "doc_name filing", "n_resamples": N_RESAMPLES, "seed": BOOTSTRAP_SEED,
    "fa_rate_denominator": "TA + FA + TR + FR", "filings": filing_count,
    "source_hashes": {model: value["provenance"]["sha256"] for model, value in evaluations.items()},
}, indent=2) + "\n")
for model, run_evaluation in evaluations.items():
    live_path = RUNS[model] / "results.jsonl"
    if hashlib.sha256(live_path.read_bytes()).hexdigest() != run_evaluation["provenance"]["sha256"]["results"]:
        raise ValueError(f"{model}'s log changed during evaluation; rerun the notebook")
    write_outputs(run_evaluation, primary_view=SCORING_VIEW)
    print(f"{model}: refreshed metrics from the latest complete run.")
for model, state in run_states.items():
    if state["state"] == "Pending":
        print(f"{model}: Pending {state['progress']}; no metrics or run summaries written.")
print("Saved statistical intervals and per-model provenance.")


Claude Haiku 4.5: refreshed metrics from the latest complete run.
Qwen2.5 7B: refreshed metrics from the latest complete run.
Qwen3 30B: refreshed metrics from the latest complete run.
Llama 3.1 8B: refreshed metrics from the latest complete run.
GPT-5.5: refreshed metrics from the latest complete run.
Fin-o1 8B: refreshed metrics from the latest complete run.
Saved statistical intervals and per-model provenance.


## Qwen3 30B — source review of four false accepts

The 16 October 2026 review corrected two missing currency units, a basic-versus-diluted EPS gold-row error, and an underspecified department in a revenue-share question. The four saved answers now receive source-reviewed credit. Everbridge remains an **ambiguity adjudication**: the question does not say which of three departments it refers to.

The shared [normalizations](tatdqa_scoring_normalizations.json) and [adjudications](tatdqa_scoring_adjudications.json) apply to every model and verdict. The original benchmark gold and saved predictions are unchanged. [Audit evidence and before/after metrics](../results/new_submissions/tatdqa_qwen3_30b/false_accept_source_review.json) include source locations, hashes, and policy limitations. This review covers these four flags, not every accepted claim.

The [results notebook](tatdqa_results.ipynb) displays the four saved answers and their current grades.


## FinanceBench — latest GPT-5.5 run (67 questions)

The completed [GPT-5.5 run](../results/new_submissions/financebench_gpt5.5/results.jsonl) is scored from saved claims with the shared source-reviewed scorer. Precision and decision accuracy use graded decisions; unresolved accepts and rejects are displayed separately. Coverage includes every raw VERIFIED or VIOLATED decision. This FinanceBench section is excluded from TADQA aggregates.


In [8]:
# Calculate scores from frozen claims; summary.json and cached grades are unused.
import hashlib
import importlib
import json
from IPython.display import Markdown
from scripts.reviewer_tatdqa import latest_run_state

financebench_scoring = importlib.reload(importlib.import_module("verifiqa.eval.financebench_scoring"))
financebench_paths = {
    "dataset": "data/numerical_questions.jsonl",
    "results": "results/new_submissions/financebench_gpt5.5/results.jsonl",
    "manifest": "results/new_submissions/financebench_gpt5.5/run_manifest.json",
    "review": "data/financebench_scoring_review.json",
    "scorer": "src/verifiqa/eval/financebench_scoring.py",
    "candidate_scorer": "src/verifiqa/eval/candidate_scoring.py",
    "unit_conversion": "src/verifiqa/verification/money_units.py",
}
financebench_table = None
financebench_result_path = ROOT / financebench_paths["results"]
if not financebench_result_path.is_file():
    print("FinanceBench: Pending 0/67; no metrics calculated or exported.")
else:
    financebench_scorer = financebench_scoring.FinanceBenchScorer.from_root(ROOT)
    financebench_ids = set(financebench_scorer.dataset)
    financebench_state = latest_run_state(
        ROOT, financebench_result_path.parent, financebench_ids, allow_incomplete=True)
    if financebench_state["state"] == "Pending":
        print(f"FinanceBench: Pending {financebench_state['progress']}; no metrics calculated or exported.")
    else:
        if not (ROOT / financebench_paths["manifest"]).is_file():
            raise ValueError("Selected GPT-5.5 FinanceBench run is missing its manifest")
        financebench_hashes = {name: hashlib.sha256((ROOT / path).read_bytes()).hexdigest()
                              for name, path in financebench_paths.items()}
        financebench_records = [json.loads(line) for line in financebench_result_path.read_text().splitlines()
                                if line.strip()]
        financebench_manifest = json.loads((ROOT / financebench_paths["manifest"]).read_text())
        financebench_metadata = financebench_manifest.get("llm_run", {})
        financebench_settings = financebench_manifest.get("settings", {})
        if (financebench_metadata.get("model") != "gpt-5.5"
                or financebench_metadata.get("mode") != "openai"
                or financebench_settings.get("model") != "gpt-5.5"
                or financebench_settings.get("mode") != "openai"
                or financebench_settings.get("claim_source") != "generated_answer"
                or financebench_settings.get("deterministic_smt") is not True
                or any(row.get("llm_run") != financebench_metadata for row in financebench_records)
                or any("baseline" in row for row in financebench_records)):
            raise ValueError("Selected FinanceBench run does not identify the GPT-5.5 verification pipeline")
        financebench_summary = financebench_scorer.summarize_rows(financebench_records)
        financebench_scores = financebench_scorer.score_rows(financebench_records)
        financebench_authority = financebench_scorer.authority_audit(financebench_result_path)
        financebench_counts = {key: financebench_summary[key] for key in
            ("TA", "FA", "TR", "FR", "AB", "UNSCORED_VERIFIED", "UNSCORED_REJECTED")}
        assert sum(financebench_counts.values()) == len(financebench_ids) == 67
        financebench_table = pd.DataFrame([{
            "Scoring": "Source reviewed (post hoc)", "Total": financebench_summary["total"],
            "VERIFIED": financebench_summary["verified"], "VIOLATED": financebench_summary["violated"],
            "ABSTAIN": financebench_summary["abstain"],
            **{key: financebench_summary[key] for key in ("TA", "FA", "TR", "FR")},
            "Unscored VERIFIED": financebench_summary["unscored_verified"],
            "Unscored VIOLATED": financebench_summary["unscored_rejected"],
            "Precision": financebench_summary["verification_precision"],
            "Decision accuracy": financebench_summary["decision_accuracy"],
            "Coverage": financebench_summary["coverage"],
        }]).set_index("Scoring")
        display(financebench_table.style.format({
            **{name: "{:.0f}" for name in financebench_table.columns
               if name not in ("Precision", "Decision accuracy", "Coverage")},
            **{name: "{:.2%}" for name in ("Precision", "Decision accuracy", "Coverage")},
        }).set_caption("FinanceBench · GPT-5.5 · source reviewed; precision and accuracy use graded decisions"))
        financebench_graded_accepts = financebench_summary["TA"] + financebench_summary["FA"]
        financebench_bounds = financebench_summary["precision_bounds"]
        display(Markdown(
            f"**Confirmed FA: {financebench_summary['FA']}.** "
            f"Precision: **{financebench_summary['TA']}/{financebench_graded_accepts}** graded accepts; "
            f"**{financebench_summary['unscored_verified']} of {financebench_summary['verified']}** "
            f"raw accepts unresolved. All-acceptance precision bounds: "
            f"**{financebench_bounds[0]:.2%}–{financebench_bounds[1]:.2%}** (not a confidence interval). "
            f"Decision accuracy: **{financebench_summary['TA'] + financebench_summary['TR']}/"
            f"{financebench_summary['scored_decisions']}** graded decisions."))
        display(Markdown(financebench_authority["display_note"]))
        financebench_outcomes = [{
            "id": record["id"], "status": record["status"],
            "claimed_value": record.get("claimed_value"),
            "claimed_value_unit": record.get("claimed_value_unit", ""),
            "gold_answer": record["raw_answer"],
            "source_reviewed_outcome": score["outcome"],
            "source_reviewed_candidate_correct": score["candidate_correct"],
            "reason": score["reason"], "reference_targets": json.dumps(score["reference_targets"]),
            "comparison_claim_unit": score["comparison_claim_unit"],
            "unit_interpretation": json.dumps(score["unit_interpretation"]),
        } for record, score in zip(financebench_records, financebench_scores)]
        if (financebench_hashes["results"] != financebench_state["sha256"] or
                any(hashlib.sha256((ROOT / path).read_bytes()).hexdigest() != financebench_hashes[name]
                    for name, path in financebench_paths.items())):
            raise ValueError("FinanceBench inputs changed during scoring; rerun this cell")
        financebench_export = ROOT / "artifacts/financebench_notebook_latest"
        financebench_export.mkdir(parents=True, exist_ok=True)
        financebench_table.to_csv(financebench_export / "results_table.csv")
        pd.DataFrame(financebench_outcomes).to_csv(financebench_export / "scored_rows.csv", index=False)
        (financebench_export / "provenance.json").write_text(json.dumps({
            "dataset": "FinanceBench", "questions": len(financebench_ids),
            "models": sorted({row.get("llm_run", {}).get("model", "unspecified")
                              for row in financebench_records}),
            "included_in_tadqa_aggregates": False,
            "scoring": financebench_summary["scoring"], "authority_audit": financebench_authority,
            "input_files": {name: {"path": path, "sha256": financebench_hashes[name]}
                            for name, path in financebench_paths.items()},
            "counts": financebench_counts,
            "precision_denominator": "TA + FA", "accuracy_denominator": "TA + FA + TR + FR",
            "raw_verified": financebench_summary["verified"],
            "graded_verified": financebench_graded_accepts,
            "precision_bounds": financebench_bounds,
        }, indent=2) + "\n")


,Total,VERIFIED,VIOLATED,ABSTAIN,TA,FA,TR,FR,Unscored VERIFIED,Unscored VIOLATED,Precision,Decision accuracy,Coverage
Scoring,,,,,,,,,,,,,
Source reviewed (post hoc),67,35,12,20,34,0,5,6,1,1,100.00%,86.67%,70.15%


**Confirmed FA: 0.** Precision: **34/34** graded accepts; **1 of 35** raw accepts unresolved. All-acceptance precision bounds: **97.14%–100.00%** (not a confidence interval). Decision accuracy: **39/45** graded decisions.

Grounding defects have not been audited for this run.